# LegalQA smoke Version 4 — tái sử dụng output Version 3

Notebook này dùng lại **407.107 embeddings**, FAISS index và model weights từ output Version 3 (scriptVersionId=348583427), rồi chạy lại 30 câu dev với retrieval đầy đủ.

Version 4 không build index và không tải lại model. Trước khi Run All, trong **Add Input → Datasets**, gắn hai dataset `lighth/ver3-smoke-output` và `lighth/uit-dsc-2026-task2-legalqa-train`. Bật GPU và Internet để clone code/cài dependencies.

## 1. Cấu hình Version 4

Artifacts Version 3 phải xuất hiện tại `/kaggle/input/ver3-smoke-output/legalqa_smoke_full_v1`. Dữ liệu train/test được đọc từ `/kaggle/input/uit-dsc-2026-task2-legalqa-train`. Output mới được ghi riêng vào `/kaggle/working/legalqa_smoke_full_retrieval_v4`.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

if not Path('/kaggle').exists():
    raise RuntimeError('Notebook smoke này chỉ được cấu hình để chạy trên Kaggle.')

VERSION3_URL = 'https://www.kaggle.com/datasets/lighth/ver3-smoke-output'
VERSION3_ROOT = Path('/kaggle/input/ver3-smoke-output/legalqa_smoke_full_v1')
SAVED_INDEX = VERSION3_ROOT / 'index'
SAVED_MODELS = VERSION3_ROOT / 'models'

REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
REPO_REF = 'main'
WORK_BASE = Path('/kaggle/working')
CODE = WORK_BASE / 'uit-dsc-2026-task2-legalqa'
RUN_NAME = 'legalqa_smoke_full_retrieval_v4'
RUN_ROOT = WORK_BASE / RUN_NAME

USE_REPO_DATA = False
KAGGLE_DATASET_ROOT = Path('/kaggle/input/uit-dsc-2026-task2-legalqa-train')
SMOKE_QUESTIONS = 30
GENERATION_MODE = 'generate'

MODELS = RUN_ROOT / 'models'  # Chỉ chứa lock/audit và symlink tới weights Version 3.
INDEX = SAVED_INDEX           # Đọc trực tiếp full index Version 3; không sao chép, không rebuild.
DATA = RUN_ROOT / 'data'
CFG = RUN_ROOT / 'smoke_config.json'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Version 3 input:', VERSION3_ROOT)
print('Version 4 output:', RUN_ROOT)

## 2. Clone code và nối artifact Version 3

Cell này kiểm tra đủ index/model của Version 3. Model weights được nối bằng symlink vào /kaggle/working để CLI có thể ghi file audit mới mà không sao chép nhiều GB dữ liệu.

In [ ]:
if CODE.exists():
    if not (CODE / '.git').is_dir():
        raise RuntimeError(f'{CODE} đã tồn tại nhưng không phải Git repo. Hãy Restart Session hoặc đổi CODE.')
    remote = subprocess.check_output(['git', '-C', str(CODE), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote.rstrip('/') != REPO_URL.rstrip('/'):
        raise RuntimeError(f'Remote không đúng repo yêu cầu: {remote}')
    subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only', 'origin', REPO_REF], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(CODE)], check=True)

required_index = ['index_manifest.json', 'corpus.sqlite', 'dense.faiss']
missing_index = [name for name in required_index if not (SAVED_INDEX / name).is_file()]
required_models = ['models.lock.json']
missing_models = [name for name in required_models if not (SAVED_MODELS / name).is_file()]
missing_roles = [role for role in ['embedding', 'reranker', 'generator'] if not (SAVED_MODELS / role / 'config.json').is_file()]
if missing_index or missing_models or missing_roles:
    raise FileNotFoundError(
        'Thiếu artifacts Version 3. Hãy Add Input dataset lighth/ver3-smoke-output từ ' + VERSION3_URL
        + f' | index={missing_index}, model_files={missing_models}, model_roles={missing_roles}'
    )

MODELS.mkdir(parents=True, exist_ok=True)
for role in ['embedding', 'reranker', 'generator']:
    source_dir = SAVED_MODELS / role
    link = MODELS / role
    if link.exists() or link.is_symlink():
        if link.resolve() != source_dir.resolve():
            raise RuntimeError(f'Symlink model không đúng: {link} -> {link.resolve()}')
    else:
        link.symlink_to(source_dir, target_is_directory=True)
shutil.copy2(SAVED_MODELS / 'models.lock.json', MODELS / 'models.lock.json')

DATASET_ROOT = CODE if USE_REPO_DATA else KAGGLE_DATASET_ROOT
TRAIN_PATH = DATASET_ROOT / 'train.json'
TEST_PATH = DATASET_ROOT / 'public-official.json'
for label, data_path in [('train', TRAIN_PATH), ('test', TEST_PATH)]:
    if not data_path.is_file():
        raise FileNotFoundError(f'{label}: {data_path}')

smoke_cfg = json.loads((CODE / 'config.json').read_text(encoding='utf-8'))
smoke_cfg['retrieval'].update({
    'bm25_k': 100,
    'dense_k': 100,
    'rrf_constant': 60,
    'pool_k': 24,
    'max_children_per_parent': 2,
    'parents_k': 4,
    'embedding_batch': 32,
    'reranker_batch': 8,
    'reranker_max_tokens': 768,
})
smoke_cfg['generation'].update({
    'max_input_tokens': 4096,
    'max_new_tokens': 1024,
    'parent_max_tokens': 1400,
    'min_context_tokens': 256,
})
CFG.write_text(json.dumps(smoke_cfg, ensure_ascii=False, indent=2), encoding='utf-8')

def run(*args):
    command = [sys.executable, '-m', 'legalqa', '--config', str(CFG), '--models', str(MODELS), *map(str, args)]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=CODE, check=True)

commit = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Code:', CODE, '| Commit:', commit)
print('Saved index:', INDEX)
print('Model links:', {role: str((MODELS / role).resolve()) for role in ['embedding', 'reranker', 'generator']})

## 3. Cài dependencies và kiểm định CPU

Dependencies vẫn được cài trong Version 4, nhưng model weights không được tải lại. Cell phải kết thúc với unit test OK và metric self-test thành công.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'nltk.downloader', '-q', 'wordnet', 'omw-1.4'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=CODE, check=True)
subprocess.run([sys.executable, 'scripts/check_metrics.py'], cwd=CODE, check=True)
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RUN_ROOT / 'environment.freeze.txt').write_text(freeze, encoding='utf-8')

## 4. Prepare lại đúng split dữ liệu

Split dùng cùng seed nên 30 ID dev phải trùng Version 3. Dữ liệu và kết quả Version 4 được ghi vào run mới để không đụng cache cũ.

In [ ]:
run('prepare', '--train', TRAIN_PATH, '--test', TEST_PATH, '--output', DATA)
report = json.loads((DATA / 'data_report.json').read_text(encoding='utf-8'))
assert report['split_sizes']['dev30'] == 30
print(json.dumps(report, ensure_ascii=False, indent=2))

## 5. Audit model tái sử dụng

CLI đọc weights qua symlink từ Version 3 và ghi parameter_audit.json mới trong output Version 4.

In [ ]:
run('audit-models')
audit = json.loads((MODELS / 'parameter_audit.json').read_text(encoding='utf-8'))
assert audit['passes'] and audit['total_with_unmerged_lora'] < 4_000_000_000
print('Parameter audit OK:', f"{audit['total_with_unmerged_lora']:,}")

## 6. Xác nhận full index Version 3

Cell này chỉ đọc manifest. Nếu không thấy đúng 407.107 chunks, notebook dừng thay vì âm thầm build lại embeddings.

In [ ]:
manifest = json.loads((INDEX / 'index_manifest.json').read_text(encoding='utf-8'))
if manifest.get('chunks') != 407_107:
    raise RuntimeError(f"Không phải full index Version 3: chunks={manifest.get('chunks')}")
if manifest.get('documents') != 8_507:
    raise RuntimeError(f"Số documents không khớp Version 3: {manifest.get('documents')}")
print('Reusing completed Version 3 index:', INDEX)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

## 7. Retrieval đầy đủ → generate → evaluate 30 câu

Retrieval cache được tạo mới vì các tham số đã tăng lên BM25/Dense 100, pool 24 và 4 parent contexts. Generation giữ giới hạn 1024 token.

In [ ]:
dev_questions = json.loads((DATA / 'dev30.questions.json').read_text(encoding='utf-8'))
dev_references = json.loads((DATA / 'dev30.references.json').read_text(encoding='utf-8'))
smoke_ids = list(dev_questions)[:SMOKE_QUESTIONS]
if not smoke_ids:
    raise RuntimeError('Dev smoke không có câu hỏi.')
SMOKE_QUESTIONS_PATH = RUN_ROOT / 'smoke.questions.json'
SMOKE_REFERENCES_PATH = RUN_ROOT / 'smoke.references.json'
SMOKE_RETRIEVAL = RUN_ROOT / 'smoke.retrieval.json'
SMOKE_PREDICTIONS = RUN_ROOT / 'smoke.predictions.json'
SMOKE_METRICS = RUN_ROOT / 'smoke.metrics.json'
SMOKE_QUESTIONS_PATH.write_text(json.dumps({key: dev_questions[key] for key in smoke_ids}, ensure_ascii=False, indent=2), encoding='utf-8')
SMOKE_REFERENCES_PATH.write_text(json.dumps({key: dev_references[key] for key in smoke_ids}, ensure_ascii=False, indent=2), encoding='utf-8')

run('retrieve', '--questions', SMOKE_QUESTIONS_PATH, '--index', INDEX, '--output', SMOKE_RETRIEVAL)
run('generate', '--questions', SMOKE_QUESTIONS_PATH, '--retrieval', SMOKE_RETRIEVAL, '--output', SMOKE_PREDICTIONS, '--mode', GENERATION_MODE)
run('evaluate', '--predictions', SMOKE_PREDICTIONS, '--references', SMOKE_REFERENCES_PATH, '--output', SMOKE_METRICS, '--label', 'pipeline_smoke')

predictions = json.loads(SMOKE_PREDICTIONS.read_text(encoding='utf-8'))
metrics = json.loads(SMOKE_METRICS.read_text(encoding='utf-8'))
print('SMOKE PASSED:', json.dumps(metrics, ensure_ascii=False, indent=2))
for key in smoke_ids[:3]:
    print('\nID:', key)
    print('Question:', dev_questions[key]['question'])
    print('Prediction:', predictions[key]['answer'])
    print('Reference:', dev_references[key])

## Kết quả mong đợi

Version 4 đạt khi cell cuối in SMOKE PASSED, xử lý đủ 30 ID và không có exception. Artifact mới nằm trong /kaggle/working/legalqa_smoke_full_retrieval_v4; full index và weights vẫn được đọc từ Version 3.